# exp/r2-v7 on Colab (T4): queue [gru_v7a, gru_v5b_s2]
Runtime -> Change runtime type -> T4 GPU, then Run all. After a disconnect, just Run all again: each config resumes from its `last.pt` on Drive (`--resume`), and a config whose `train_log.csv` already reached the final epoch or has an `early_stop` row is skipped. Each run writes `best.pt` (best validation PESQ, inference weights), `last.pt` (full training state, every epoch) and `train_log.csv` to its Drive checkpoint dir; console output is appended to `train_stdout.log` there.

- **gru_v7a**: complex ratio mask trained **from scratch** (random init, seed 0, no warm start), mask_imag_scale 1.0, gru_v5b architecture and remix/gain augmentation. Loss = 0.7 compressed magnitude + 0.3 compressed complex (Braun & Tashev 2021, c = 0.3, alpha = 0.3) + 0.5 MR-STFT + 0.3 (-SI-SDR). lr 1e-3 (gru_v2b's from-scratch lr), 300 epochs, early stop only after 75 val points (150 epochs) without gain. Estimated T4 time: ~16 s/epoch incl. validation, so ~80 min for 300 epochs (cannot stop before epoch ~152, ~40 min).
- **gru_v5b_s2**: seed control, identical to gru_v5b (warm start from weights/gru_v4b.pt, lr 2e-4, 100 epochs, early stop after 8 val points) except seed 2 (gru_v5b: 0). Estimated T4 time: ~15 s/epoch, ~6 min if it stops like gru_v5b (epoch 24), at most ~26 min.

Estimates come from logs/train_log_v5b.csv (T4: 8.5 s/train epoch + 13.5 s per validation every 2 epochs) and gru_v6a (crm, ~9 s/epoch); add ~5 min setup (clone, pip, data copy).

Drive layout: dataset at `MyDrive/noise_reduction_challenge/{train,validation}/...`; checkpoints go to `MyDrive/nrc_ckpt/<config_stem>/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
!rm -rf /content/repo && git clone -b exp/r2-v7 --depth 1 https://github.com/pauljh4323/SGSYS_13thTeam_NoiseReductionChallenge.git /content/repo
%cd /content/repo
assert os.path.getsize('weights/gru_v4b.pt') > 1_000_000, 'weights/gru_v4b.pt missing (gru_v5b_s2 init_from)'
print('init weights OK:', os.path.getsize('weights/gru_v4b.pt'), 'bytes; CPUs:', os.cpu_count())


In [ ]:
!pip install -q pesq==0.0.4 pystoi soundfile pyyaml

In [ ]:
!mkdir -p /content/data && cp -rn /content/drive/MyDrive/noise_reduction_challenge /content/data/

In [ ]:
import os, yaml, pandas as pd
from pathlib import Path
CONFIGS = ["configs/dl/gru_v7a.yaml", "configs/dl/gru_v5b_s2.yaml"]  # run in this order
WORKERS = os.cpu_count() or 2  # loader workers (only used by noise-aware models)
CKPT_ROOT = Path('/content/drive/MyDrive/nrc_ckpt')

def ckpt_dir(cfg): return CKPT_ROOT / Path(cfg).stem

def finished(cfg):
    log = ckpt_dir(cfg) / 'train_log.csv'
    epochs = yaml.safe_load(open(cfg))['train']['epochs']
    if not log.is_file() or os.path.getsize(log) == 0:
        return False
    df = pd.read_csv(log)
    if len(df) == 0:
        return False
    early = 'early_stop' in df and (pd.to_numeric(df.early_stop, errors='coerce') == 1).any()
    return bool(early) or df.epoch.max() >= epochs

for cfg in CONFIGS:
    d = ckpt_dir(cfg)
    if finished(cfg):
        print(f'skip {cfg}: final epoch reached or early-stopped')
        continue
    d.mkdir(parents=True, exist_ok=True)
    !python scripts/train.py --config {cfg} --data-root /content/data/noise_reduction_challenge --ckpt-dir "{d}" --resume --device auto --loader-workers {WORKERS} 2>&1 | tee -a "{d}/train_stdout.log"

In [ ]:
for cfg in CONFIGS:
    log = ckpt_dir(cfg) / 'train_log.csv'
    if not log.is_file():
        print(f'{Path(cfg).stem}: no train_log.csv'); continue
    df = pd.read_csv(log).dropna(subset=['val_pesq'])
    if df.empty:
        print(f'{Path(cfg).stem}: no validation yet'); continue
    r = df.loc[df.val_pesq.idxmax()]
    print(f'{Path(cfg).stem}: best epoch {int(r.epoch)}, best val PESQ {r.val_pesq:.4f} (last epoch {int(pd.read_csv(log).epoch.max())})')